In [1]:
import os
import json
import pandas as pd
import numpy as np

# ============================================================
# PROJECT ROOT (LOCKED)
# ============================================================

PROJECT_ROOT = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair"

PROFILE_PATH = os.path.join(PROJECT_ROOT, "reports/profiling_reports/dataset_profile.csv")

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "reports/issue_reports")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "detected_issues.csv")

CONFIG_PATH = os.path.join(PROJECT_ROOT, "notebooks", "config", "runtime_config.json")

WORKING_OUTPUT_PATH = os.path.join(PROJECT_ROOT, "data", "working", "df_working.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# FAIL-FAST INPUT CHECK
# ============================================================

if not os.path.exists(PROFILE_PATH):
    raise FileNotFoundError(f"Missing dataset_profile: {PROFILE_PATH}")

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Missing runtime_config: {CONFIG_PATH}")

In [2]:
# =========================
# LOAD DATASET FROM CONFIG (RESTORE THIS)
# =========================

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

if "input_path" not in config:
    raise ValueError("runtime_config.json missing 'input_path'")

DATA_PATH = config["input_path"]

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(DATA_PATH)

# normalize column names EARLY
df.columns = df.columns.str.strip().str.lower()

print("✅ Dataset loaded:", df.shape)
print("Columns:", df.columns.tolist())

✅ Dataset loaded: (891, 12)
Columns: ['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


In [3]:
def normalize_missing(df):

    MISSING_TOKENS = {
        "?", "??", "???",
        "na", "n/a", "nan",
        "null", "none",
        "unknown", "missing",
        "-", "--",
        ""
    }

    for col in df.columns:

        # convert safely
        series = df[col].astype("string")

        # normalize
        series = series.str.strip().str.lower()

        # 🔥 universal replacement
        series[series.isin(MISSING_TOKENS)] = np.nan

        df[col] = series

    return df

print("\n SAMPLE COLUMN VALUES:")

for col in df.columns[:5]:   # show first 5 columns only
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False).head(5))


 SAMPLE COLUMN VALUES:

Column: passengerid
passengerid
1    1
2    1
3    1
4    1
5    1
Name: count, dtype: int64

Column: survived
survived
0    549
1    342
Name: count, dtype: int64

Column: pclass
pclass
3    491
1    216
2    184
Name: count, dtype: int64

Column: name
name
Braund, Mr. Owen Harris                                1
Cumings, Mrs. John Bradley (Florence Briggs Thayer)    1
Heikkinen, Miss. Laina                                 1
Futrelle, Mrs. Jacques Heath (Lily May Peel)           1
Allen, Mr. William Henry                               1
Name: count, dtype: int64

Column: sex
sex
male      577
female    314
Name: count, dtype: int64


In [4]:
# =========================
# APPLY NORMALIZATION
# =========================
df = normalize_missing(df)

print("\nAFTER NORMALIZATION:")
print(df.isna().sum().sort_values(ascending=False).head(10))


AFTER NORMALIZATION:
cabin          687
age            177
embarked         2
passengerid      0
name             0
pclass           0
survived         0
sex              0
parch            0
sibsp            0
dtype: int64


In [5]:
dataset_profile = pd.read_csv(PROFILE_PATH)

if dataset_profile.empty:
    raise ValueError("dataset_profile is empty")

In [6]:
required_base = [
    "column_name",
    "null_ratio",
    "unique_ratio",
    "dtype"
]

missing_base = [col for col in required_base if col not in dataset_profile.columns]

if missing_base:
    raise ValueError(f"Missing required base columns: {missing_base}")

# derive n_unique if needed
if "n_unique" not in dataset_profile.columns:
    if "unique_count" in dataset_profile.columns:
        dataset_profile["n_unique"] = dataset_profile["unique_count"]
    elif "row_count" in dataset_profile.columns:
        dataset_profile["n_unique"] = (
            dataset_profile["unique_ratio"] * dataset_profile["row_count"]
        ).astype(int)
    else:
        raise ValueError("Cannot derive n_unique")

required_final = [
    "column_name",
    "null_ratio",
    "unique_ratio",
    "n_unique",
    "dtype"
]

missing_final = [col for col in required_final if col not in dataset_profile.columns]

if missing_final:
    raise ValueError(f"Missing required columns: {missing_final}")

print("✅ dataset_profile validated")

✅ dataset_profile validated


In [7]:
issues = []

def add_issue(col, issue):
    issues.append({
        "column_name": col,
        "issue_type": issue
    })

for _, row in dataset_profile.iterrows():

    col = str(row["column_name"]).strip().lower()
    col_lower = col.lower()

    unique_ratio = float(row["unique_ratio"])
    dtype = str(row["dtype"])

    # 🔥 CRITICAL: USE REAL DATA
    if col not in df.columns:
        continue

    # =========================
    # TRUE MISSING DETECTION
    # =========================
    actual_missing = df[col].isna().sum()

    if actual_missing > 0:
        add_issue(col, "missing_values")

    # =========================
    # HIGH CARDINALITY
    # =========================
    if unique_ratio > 0.9:
        add_issue(col, "high_cardinality")

    # =========================
    # ID COLUMN
    # =========================
    if "id" in col_lower:
        add_issue(col, "id_column")

In [8]:
detected_issues = pd.DataFrame(issues)

if detected_issues.empty:
    raise ValueError("No issues detected")

detected_issues = detected_issues[["column_name", "issue_type"]]

if detected_issues.isnull().any().any():
    raise ValueError("Null values found")

detected_issues = detected_issues.drop_duplicates()

detected_issues = detected_issues.sort_values(
    by=["column_name", "issue_type"]
).reset_index(drop=True)

print("✅ detected_issues built")

✅ detected_issues built


In [9]:
detected_issues.to_csv(OUTPUT_PATH, index=False)

print("✅ detected_issues.csv saved")
print(OUTPUT_PATH)

✅ detected_issues.csv saved
/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/issue_reports/detected_issues.csv


In [10]:
df.to_csv(WORKING_OUTPUT_PATH, index=False)

print("✅ df_working saved correctly")
print("Path:", WORKING_OUTPUT_PATH)

✅ df_working saved correctly
Path: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/data/working/df_working.csv
